In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
transactions = pd.read_csv('data/transactions_train.csv')
customers = pd.read_csv('data/customers.csv')
articles = pd.read_csv('data/articles.csv')

print("Transactions:", transactions.shape)
print("Customers:", customers.shape)
print("Articles:", articles.shape)

Fontconfig warning: ignoring UTF-8: not a valid region tag
Matplotlib is building the font cache; this may take a moment.


Transactions: (31788324, 5)
Customers: (1371980, 7)
Articles: (105542, 25)


In [2]:
print("=== TRANSACTIONS ===")
print(transactions.head())
print("\n=== CUSTOMERS ===")
print(customers.head())
print("\n=== ARTICLES ===")
print(articles.head())

=== TRANSACTIONS ===
        t_dat                                        customer_id  article_id  \
0  2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   663713001   
1  2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   541518023   
2  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   505221004   
3  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   685687003   
4  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   685687004   

      price  sales_channel_id  
0  0.050831                 2  
1  0.030492                 2  
2  0.015237                 2  
3  0.016932                 2  
4  0.016932                 2  

=== CUSTOMERS ===
                                         customer_id   FN  Active  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  NaN     NaN   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  NaN     NaN   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  NaN     NaN   
3  00005ca1

In [3]:
print("=== TRANSACTIONS ===")
print(transactions.dtypes)
print(transactions.isnull().sum())

print("\n=== CUSTOMERS ===")
print(customers.dtypes)
print(customers.isnull().sum())

print("\n=== ARTICLES ===")
print(articles.dtypes)
print(articles.isnull().sum())

=== TRANSACTIONS ===
t_dat                   str
customer_id             str
article_id            int64
price               float64
sales_channel_id      int64
dtype: object
t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

=== CUSTOMERS ===
customer_id                   str
FN                        float64
Active                    float64
club_member_status            str
fashion_news_frequency        str
age                       float64
postal_code                   str
dtype: object
customer_id                    0
FN                        895050
Active                    907576
club_member_status          6062
fashion_news_frequency     16011
age                        15861
postal_code                    0
dtype: int64

=== ARTICLES ===
article_id                      int64
product_code                    int64
prod_name                         str
product_type_no                 int64
product_type_name

In [6]:
# Convert date to datetime
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print("Date range:", transactions['t_dat'].min(), "to", transactions['t_dat'].max())
print("Unique customers:", transactions['customer_id'].nunique())
print("Unique articles:", transactions['article_id'].nunique())
print("\nSales channel breakdown:")
print(transactions['sales_channel_id'].value_counts())
print("\nPrice stats:")
print(transactions['price'].describe())

# Save the cleaned transactions with parsed dates
transactions.to_csv('data/transactions_clean.csv', index=False)
print("Saved!")

Date range: 2018-09-20 00:00:00 to 2020-09-22 00:00:00
Unique customers: 1362281
Unique articles: 104547

Sales channel breakdown:
sales_channel_id
2    22379862
1     9408462
Name: count, dtype: int64

Price stats:
count    3.178832e+07
mean     2.782927e-02
std      1.918113e-02
min      1.694915e-05
25%      1.581356e-02
50%      2.540678e-02
75%      3.388136e-02
max      5.915254e-01
Name: price, dtype: float64
Saved!


In [7]:
print("Age distribution:")
print(customers['age'].describe())

print("\nClub member status:")
print(customers['club_member_status'].value_counts())

print("\nFashion news frequency:")
print(customers['fashion_news_frequency'].value_counts())

print("\nFN (fashion news subscriber) breakdown:")
print(customers['FN'].value_counts(dropna=False))

print("\nActive flag breakdown:")
print(customers['Active'].value_counts(dropna=False))

Age distribution:
count    1.356119e+06
mean     3.638696e+01
std      1.431363e+01
min      1.600000e+01
25%      2.400000e+01
50%      3.200000e+01
75%      4.900000e+01
max      9.900000e+01
Name: age, dtype: float64

Club member status:
club_member_status
ACTIVE        1272491
PRE-CREATE      92960
LEFT CLUB         467
Name: count, dtype: int64

Fashion news frequency:
fashion_news_frequency
NONE         877711
Regularly    477416
Monthly         842
Name: count, dtype: int64

FN (fashion news subscriber) breakdown:
FN
NaN    895050
1.0    476930
Name: count, dtype: int64

Active flag breakdown:
Active
NaN    907576
1.0    464404
Name: count, dtype: int64


In [8]:
# Monthly revenue trend
monthly = transactions.groupby(transactions['t_dat'].dt.to_period('M')).agg(
    num_transactions=('article_id', 'count'),
    num_customers=('customer_id', 'nunique'),
    total_revenue=('price', 'sum')
).reset_index()
monthly['t_dat'] = monthly['t_dat'].astype(str)

monthly.to_csv('data/monthly_summary.csv', index=False)
print("Monthly summary:", monthly.shape)
print(monthly.head())

Monthly summary: (25, 4)
     t_dat  num_transactions  num_customers  total_revenue
0  2018-09            594776         140340   17755.639593
1  2018-10           1397040         274220   41584.816610
2  2018-11           1270619         274207   39000.988390
3  2018-12           1148827         254059   32532.514712
4  2019-01           1263471         242024   33449.981475


In [9]:
# Customer purchase frequency segments
customer_freq = transactions.groupby('customer_id').agg(
    num_purchases=('article_id', 'count'),
    total_spend=('price', 'sum'),
    first_purchase=('t_dat', 'min'),
    last_purchase=('t_dat', 'max')
).reset_index()

def freq_segment(n):
    if n == 1:
        return '1 purchase'
    elif n <= 5:
        return '2-5 purchases'
    elif n <= 10:
        return '6-10 purchases'
    else:
        return '10+ purchases'

customer_freq['segment'] = customer_freq['num_purchases'].apply(freq_segment)
customer_freq.to_csv('data/customer_frequency.csv', index=False)
print("Customer frequency:", customer_freq.shape)
print(customer_freq['segment'].value_counts())

Customer frequency: (1362281, 6)
segment
10+ purchases     641782
2-5 purchases     369844
6-10 purchases    219141
1 purchase        131514
Name: count, dtype: int64


In [10]:
# Channel breakdown by month
channel_monthly = transactions.groupby([
    transactions['t_dat'].dt.to_period('M'),
    'sales_channel_id'
]).agg(
    num_transactions=('article_id', 'count'),
    total_revenue=('price', 'sum')
).reset_index()
channel_monthly['t_dat'] = channel_monthly['t_dat'].astype(str)
channel_monthly['channel'] = channel_monthly['sales_channel_id'].map({1: 'In-Store', 2: 'Online'})
channel_monthly.to_csv('data/channel_monthly.csv', index=False)
print("Channel monthly:", channel_monthly.shape)

Channel monthly: (49, 5)


In [11]:
# Top product categories - merge articles
top_categories = transactions.merge(articles[['article_id', 'product_group_name', 'product_type_name']], on='article_id', how='left')
category_summary = top_categories.groupby('product_group_name').agg(
    num_transactions=('article_id', 'count'),
    total_revenue=('price', 'sum'),
    num_customers=('customer_id', 'nunique')
).reset_index().sort_values('num_transactions', ascending=False)
category_summary.to_csv('data/category_summary.csv', index=False)
print("Categories:", category_summary.shape)
print(category_summary.head(10))

Categories: (19, 4)
    product_group_name  num_transactions  total_revenue  num_customers
7   Garment Upper body          12552755  338981.789203        1092325
6   Garment Lower body           7046054  231774.620034         927480
5    Garment Full body           3552470  128478.941678         644925
15            Swimwear           2579222   57628.166508         426146
16           Underwear           2565858   54395.857458         517715
0          Accessories           1599593   24893.822424         491251
12               Shoes            745521   28888.356373         269195
13      Socks & Tights            685712    7811.136525         267096
11           Nightwear            348180    8852.815390         169597
18             Unknown             97040    2598.734051          64345


In [12]:
# Age group analysis - merge customers
transactions_with_age = transactions.merge(customers[['customer_id', 'age', 'fashion_news_frequency']], on='customer_id', how='left')

def age_bucket(age):
    if age < 25:
        return 'Under 25'
    elif age < 35:
        return '25-34'
    elif age < 45:
        return '35-44'
    elif age < 55:
        return '45-54'
    else:
        return '55+'

transactions_with_age['age_group'] = transactions_with_age['age'].apply(
    lambda x: age_bucket(x) if pd.notna(x) else 'Unknown'
)

age_summary = transactions_with_age.groupby('age_group').agg(
    num_transactions=('article_id', 'count'),
    total_revenue=('price', 'sum'),
    num_customers=('customer_id', 'nunique')
).reset_index()
age_summary.to_csv('data/age_summary.csv', index=False)
print("Age summary:", age_summary)

Age summary:   age_group  num_transactions  total_revenue  num_customers
0     25-34          11565068  324373.074186         391009
1     35-44           4111309  112716.558814         167950
2     45-54           6079556  173573.402763         252409
3       55+           3367082  101355.181712         181325
4  Under 25           6525051  168823.265712         353827
5   Unknown            140258    3804.490864          15761


In [1]:
import pandas as pd
import numpy as np

transactions = pd.read_csv('data/transactions_clean.csv', parse_dates=['t_dat'])
customers = pd.read_csv('data/customers.csv')
articles = pd.read_csv('data/articles.csv')

print("Loaded:", transactions.shape, customers.shape, articles.shape)

Loaded: (31788324, 5) (1371980, 7) (105542, 25)


In [2]:
# RFM analysis - snapshot date is day after last transaction
snapshot_date = transactions['t_dat'].max() + pd.Timedelta(days=1)

rfm = transactions.groupby('customer_id').agg(
    recency=('t_dat', lambda x: (snapshot_date - x.max()).days),
    frequency=('article_id', 'count'),
    monetary=('price', 'sum')
).reset_index()

# Score each dimension 1-4 (4 is best)
rfm['r_score'] = pd.qcut(rfm['recency'], q=4, labels=[4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

def rfm_segment(row):
    if row['rfm_score'] >= 10:
        return 'Champions'
    elif row['rfm_score'] >= 8:
        return 'Loyal Customers'
    elif row['rfm_score'] >= 6:
        return 'Potential Loyalists'
    elif row['r_score'] >= 3:
        return 'Recent Customers'
    elif row['rfm_score'] >= 4:
        return 'At Risk'
    else:
        return 'Hibernating'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)

# Join with customer attributes
rfm = rfm.merge(customers[['customer_id', 'age', 'fashion_news_frequency', 'club_member_status']], 
                on='customer_id', how='left')

rfm.to_csv('data/rfm_segments.csv', index=False)
print(rfm['segment'].value_counts())
print(rfm.shape)

segment
Champions              415930
Potential Loyalists    266557
At Risk                253952
Loyal Customers        251281
Hibernating            133027
Recent Customers        41534
Name: count, dtype: int64
(1362281, 12)


In [3]:
# Flag each transaction as new or repeat customer
first_purchase = transactions.groupby('customer_id')['t_dat'].min().reset_index()
first_purchase.columns = ['customer_id', 'first_purchase_date']

transactions_flagged = transactions.merge(first_purchase, on='customer_id')
transactions_flagged['is_repeat'] = (
    transactions_flagged['t_dat'] > transactions_flagged['first_purchase_date']
).astype(int)

repeat_monthly = transactions_flagged.groupby(
    transactions_flagged['t_dat'].dt.to_period('M')
).agg(
    total_customers=('customer_id', 'nunique'),
    repeat_customers=('customer_id', lambda x: x[transactions_flagged.loc[x.index, 'is_repeat'] == 1].nunique())
).reset_index()

repeat_monthly['repeat_rate'] = repeat_monthly['repeat_customers'] / repeat_monthly['total_customers']
repeat_monthly['t_dat'] = repeat_monthly['t_dat'].astype(str)

repeat_monthly.to_csv('data/repeat_customer_monthly.csv', index=False)
print(repeat_monthly.head())

     t_dat  total_customers  repeat_customers  repeat_rate
0  2018-09           140340             20283     0.144528
1  2018-10           274220            115479     0.421118
2  2018-11           274207            163497     0.596254
3  2018-12           254059            180446     0.710252
4  2019-01           242024            183831     0.759557


In [4]:
newsletter_rfm = rfm.groupby('fashion_news_frequency').agg(
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_rfm_score=('rfm_score', 'mean'),
    customer_count=('customer_id', 'count')
).reset_index()

newsletter_rfm.to_csv('data/newsletter_rfm.csv', index=False)
print(newsletter_rfm)

  fashion_news_frequency  avg_recency  avg_frequency  avg_monetary  \
0                Monthly   398.183333      12.282143      0.307369   
1                   NONE   251.557963      20.685726      0.579848   
2              Regularly   204.551033      28.715539      0.791457   

   avg_rfm_score  customer_count  
0       6.292857             840  
1       7.231078          871598  
2       8.056793          473843  


In [5]:
def age_bucket(age):
    if age < 25: return 'Under 25'
    elif age < 35: return '25-34'
    elif age < 45: return '35-44'
    elif age < 55: return '45-54'
    else: return '55+'

customers['age_group'] = customers['age'].apply(
    lambda x: age_bucket(x) if pd.notna(x) else 'Unknown'
)

tx_enriched = transactions.merge(
    customers[['customer_id', 'age_group']], on='customer_id', how='left'
).merge(
    articles[['article_id', 'product_group_name']], on='article_id', how='left'
)

age_category = tx_enriched.groupby(['age_group', 'product_group_name']).agg(
    num_transactions=('article_id', 'count')
).reset_index()

# Keep only top 8 categories to keep it readable
top_cats = category_summary = tx_enriched['product_group_name'].value_counts().head(8).index
age_category = age_category[age_category['product_group_name'].isin(top_cats)]

age_category.to_csv('data/age_category_heatmap.csv', index=False)
print(age_category.head(10))

   age_group  product_group_name  num_transactions
0      25-34         Accessories            592326
4      25-34   Garment Full body           1333708
5      25-34  Garment Lower body           2476866
6      25-34  Garment Upper body           4421309
11     25-34               Shoes            303789
12     25-34      Socks & Tights            260824
14     25-34            Swimwear           1011090
15     25-34           Underwear            996688
18     35-44         Accessories            197898
23     35-44   Garment Full body            483591
